In [0]:
"""
id: python_3
template: python
templateVersion: 1.0.0
name: Api_customers
position:
  x: 1082
  y: 556
description:
  text: Load data from a URL if no input is provided; otherwise, use the input data.
  hash: 78312ef6
previewCodeHash: 036baee366d04494
previewMode: "1000"
config:
  code: |
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd
        from io import StringIO
        import requests
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/customers.csv"

        response=requests.get(url)
        csv_data=response.content.decode('utf-8')
        df=pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd
        from io import StringIO
        import requests
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/customers.csv"

        response=requests.get(url)
        csv_data=response.content.decode('utf-8')
        df=pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_3.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["python_3.result"])

In [0]:
"""
id: python_4
template: python
templateVersion: 1.0.0
name: API_Shipments
position:
  x: 1084
  y: 684
description:
  text: Load data from a CSV file URL or use provided data, then create a table.
  hash: 6f67e558
previewCodeHash: e319aee6cdc4c1bb
previewMode: "1000"
config:
  code: |
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd
        from io import StringIO
        import requests
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"

        response=requests.get(url)
        csv_data=response.content.decode('utf-8')
        df=pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd
        from io import StringIO
        import requests
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"

        response=requests.get(url)
        csv_data=response.content.decode('utf-8')
        df=pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_4.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["python_4.result"])

In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: orders
position:
  x: 679
  y: 318
description:
  text: Read all data from the 'lakeflow_designer.raw.orders' table.
  hash: 642b21f6
previewCodeHash: c8be8317e5ac14d2
previewMode: "1000"
config:
  table_source:
    tableName: lakeflow_designer.raw.orders
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "lakeflow_designer.raw.orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", False):
    display(ctx["source_0.data"])

In [0]:
"""
id: source_1
template: source
templateVersion: 2.0.0
name: order_items
position:
  x: 679
  y: 463
description:
  text: Read all data from the order_items table.
  hash: 6f0d6801
previewCodeHash: 98105f5713e17686
previewMode: "1000"
config:
  table_source:
    tableName: lakeflow_designer.raw.order_items
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "lakeflow_designer.raw.order_items"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_1.data"] = out["data"]
if globals().get("ld_display_outputs", False):
    display(ctx["source_1.data"])

In [0]:
"""
id: source_14
template: source
templateVersion: 2.0.0
name: Reviews
position:
  x: 2333
  y: 821.625
description:
  text: Load all data from the reviews table.
  hash: 56ff95ce
previewCodeHash: be40bbb32105c261
previewMode: "1000"
config:
  table_source:
    tableName: lakeflow_designer.raw.reviews
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "lakeflow_designer.raw.reviews"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_14.data"] = out["data"]
if globals().get("ld_display_outputs", False):
    display(ctx["source_14.data"])

In [0]:
"""
id: join_2
template: join
templateVersion: 2.0.0
name: Orders_join_Orderitems
position:
  x: 1066
  y: 427
description:
  text: Keep all rows from the left table and matching rows from the right table based on order_id.
  hash: 8b8317ed
previewCodeHash: 005cc8029eae187c
previewMode: "1000"
config:
  join_type: left
  join_keys:
    - left: order_id
      right: order_id
  join_conditions: ""
  match_case: false
  left_columns:
    edits: []
    ordered: []
  right_columns:
    edits:
      - column: order_id
        checked: false
    ordered: []
input:
  - node: source_0
    input_port: left
    output_port: data
  - node: source_1
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Any, Dict, List
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(side: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + side + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(df_left, df_right, left_cfg: Dict[str, Any], right_cfg: Dict[str, Any]):
    left_cols = list(df_left.columns)
    right_cols = list(df_right.columns)
    left_edits = _edits_by_column(left_cfg)
    right_edits = _edits_by_column(right_cfg)
    left_lower = {c.lower() for c in left_cols}

    out = []
    for name in _ordered_names(left_cols, left_cfg):
        edit = left_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("left", name)
        alias = edit.get("alias") if edit else None
        out.append(col.alias(alias) if alias else col.alias(name))
    for name in _ordered_names(right_cols, right_cfg):
        edit = right_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("right", name)
        alias = edit.get("alias") if edit else None
        if alias:
            out.append(col.alias(alias))
        elif name.lower() in left_lower:
            out.append(col.alias("right_" + name))
        else:
            out.append(col.alias(name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_keys: List[Dict[str, str]] = config.get("join_keys", [])
    join_condition = config.get("join_conditions", "")
    join_type = config.get("join_type") or "split_join"
    match_case = config.get("match_case", False)
    left_cfg = config.get("left_columns") or {}
    right_cfg = config.get("right_columns") or {}

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    left_types = {f.name.lower(): f.dataType for f in df_left.schema}
    right_types = {f.name.lower(): f.dataType for f in df_right.schema}

    def key_col(side, name, types):
        col = _qualified_col(side, name)
        if not match_case and isinstance(types.get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    predicates = []
    for key in join_keys:
        predicates.append(
            key_col("left", key["left"], left_types)
            == key_col("right", key["right"], right_types)
        )
    if join_condition:
        predicates.append(F.expr(join_condition))

    join_expr = None
    for predicate in predicates:
        join_expr = predicate if join_expr is None else join_expr & predicate

    is_split = join_type == "split_join"
    matched_how = "inner" if is_split else join_type

    if join_expr is None:
        matched = df_left.join(df_right, how=matched_how)
    else:
        matched = df_left.join(df_right, join_expr, how=matched_how)

    projection = _projection(df_left, df_right, left_cfg, right_cfg)
    if projection:
        matched = matched.select(*projection)

    if is_split:
        if join_expr is None:
            left_unmatched = df_left.join(df_right, how="left_anti")
            right_unmatched = df_right.join(df_left, how="left_anti")
        else:
            left_unmatched = df_left.join(df_right, join_expr, how="left_anti")
            right_unmatched = df_right.join(df_left, join_expr, how="left_anti")
    else:
        left_unmatched = spark.createDataFrame([], df_left.schema)
        right_unmatched = spark.createDataFrame([], df_right.schema)

    return {
        "joined_data": matched,
        "left_unmatched": left_unmatched,
        "right_unmatched": right_unmatched,
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_keys": [
        {
            "left": "order_id",
            "right": "order_id"
        }
    ],
    "join_conditions": "",
    "match_case": False,
    "left_columns": {
        "edits": [],
        "ordered": []
    },
    "right_columns": {
        "edits": [
            {
                "column": "order_id",
                "checked": False
            }
        ],
        "ordered": []
    }
}
inputs = {
    "left": ctx["source_0.data"],
    "right": ctx["source_1.data"]
}
out = run(config, inputs, spark)
ctx["join_2.joined_data"] = out["joined_data"]
ctx["join_2.left_unmatched"] = out["left_unmatched"]
ctx["join_2.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False):
    display(ctx["join_2.joined_data"])
    display(ctx["join_2.left_unmatched"])
    display(ctx["join_2.right_unmatched"])

In [0]:
"""
id: ai_function_15
template: ai_function
templateVersion: 3.0.0
name: SentimentAnalysis
position:
  x: 2593
  y: 821.625
description:
  text: Analyze sentiment of review_body while keeping all original columns.
  hash: 5c7754f0
previewCodeHash: 035ca5cf104244c9
previewMode: "1000"
config:
  expressions:
    - ai_analyze_sentiment(review_body) `Sentiment`
  keep_all_columns: true
input:
  - node: source_14
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_analyze_sentiment(review_body) `Sentiment`"
    ],
    "keep_all_columns": True
}
inputs = {
    "data": ctx["source_14.data"]
}
out = run(config, inputs, spark)
ctx["ai_function_15.ai_data"] = out["ai_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["ai_function_15.ai_data"])

In [0]:
"""
id: join_5
template: join
templateVersion: 2.0.0
name: Ordersitems_Join_Customer
position:
  x: 1391
  y: 504
description:
  text: Perform a left join on customer_id, excluding customer_id columns from the output.
  hash: 91b182c4
previewCodeHash: d61df84bf99ea1c2
previewMode: "1000"
config:
  join_type: left
  join_keys:
    - left: customer_id
      right: customer_id
  join_conditions: ""
  match_case: false
  left_columns:
    edits:
      - column: customer_id
        checked: false
    ordered: []
  right_columns:
    edits:
      - column: customer_id
        checked: false
    ordered: []
input:
  - node: python_3
    input_port: right
    output_port: result
  - node: join_2
    input_port: left
    output_port: joined_data
"""

# generated from the system
from typing import Any, Dict, List
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(side: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + side + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(df_left, df_right, left_cfg: Dict[str, Any], right_cfg: Dict[str, Any]):
    left_cols = list(df_left.columns)
    right_cols = list(df_right.columns)
    left_edits = _edits_by_column(left_cfg)
    right_edits = _edits_by_column(right_cfg)
    left_lower = {c.lower() for c in left_cols}

    out = []
    for name in _ordered_names(left_cols, left_cfg):
        edit = left_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("left", name)
        alias = edit.get("alias") if edit else None
        out.append(col.alias(alias) if alias else col.alias(name))
    for name in _ordered_names(right_cols, right_cfg):
        edit = right_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("right", name)
        alias = edit.get("alias") if edit else None
        if alias:
            out.append(col.alias(alias))
        elif name.lower() in left_lower:
            out.append(col.alias("right_" + name))
        else:
            out.append(col.alias(name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_keys: List[Dict[str, str]] = config.get("join_keys", [])
    join_condition = config.get("join_conditions", "")
    join_type = config.get("join_type") or "split_join"
    match_case = config.get("match_case", False)
    left_cfg = config.get("left_columns") or {}
    right_cfg = config.get("right_columns") or {}

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    left_types = {f.name.lower(): f.dataType for f in df_left.schema}
    right_types = {f.name.lower(): f.dataType for f in df_right.schema}

    def key_col(side, name, types):
        col = _qualified_col(side, name)
        if not match_case and isinstance(types.get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    predicates = []
    for key in join_keys:
        predicates.append(
            key_col("left", key["left"], left_types)
            == key_col("right", key["right"], right_types)
        )
    if join_condition:
        predicates.append(F.expr(join_condition))

    join_expr = None
    for predicate in predicates:
        join_expr = predicate if join_expr is None else join_expr & predicate

    is_split = join_type == "split_join"
    matched_how = "inner" if is_split else join_type

    if join_expr is None:
        matched = df_left.join(df_right, how=matched_how)
    else:
        matched = df_left.join(df_right, join_expr, how=matched_how)

    projection = _projection(df_left, df_right, left_cfg, right_cfg)
    if projection:
        matched = matched.select(*projection)

    if is_split:
        if join_expr is None:
            left_unmatched = df_left.join(df_right, how="left_anti")
            right_unmatched = df_right.join(df_left, how="left_anti")
        else:
            left_unmatched = df_left.join(df_right, join_expr, how="left_anti")
            right_unmatched = df_right.join(df_left, join_expr, how="left_anti")
    else:
        left_unmatched = spark.createDataFrame([], df_left.schema)
        right_unmatched = spark.createDataFrame([], df_right.schema)

    return {
        "joined_data": matched,
        "left_unmatched": left_unmatched,
        "right_unmatched": right_unmatched,
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_keys": [
        {
            "left": "customer_id",
            "right": "customer_id"
        }
    ],
    "join_conditions": "",
    "match_case": False,
    "left_columns": {
        "edits": [
            {
                "column": "customer_id",
                "checked": False
            }
        ],
        "ordered": []
    },
    "right_columns": {
        "edits": [
            {
                "column": "customer_id",
                "checked": False
            }
        ],
        "ordered": []
    }
}
inputs = {
    "right": ctx["python_3.result"],
    "left": ctx["join_2.joined_data"]
}
out = run(config, inputs, spark)
ctx["join_5.joined_data"] = out["joined_data"]
ctx["join_5.left_unmatched"] = out["left_unmatched"]
ctx["join_5.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False):
    display(ctx["join_5.joined_data"])
    display(ctx["join_5.left_unmatched"])
    display(ctx["join_5.right_unmatched"])

In [0]:
"""
id: output_16
template: output
templateVersion: 3.0.0
name: lakeflow_designer.enrich.Sentiment
position:
  x: 2853
  y: 821.625
description:
  text: Overwrite data in table_17.
  hash: 0935bcff
previewMode: "1000"
config:
  output_type: table
  catalog: lakeflow_designer
  schema: enrich
  table_name: Sentiment
  write_mode: overwrite
input:
  - node: ai_function_15
    input_port: data
    output_port: ai_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "lakeflow_designer",
    "schema": "enrich",
    "table_name": "Sentiment",
    "write_mode": "overwrite"
}
inputs = {
    "data": ctx["ai_function_15.ai_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: join_6
template: join
templateVersion: 2.0.0
name: OBT
position:
  x: 1651.333295583725
  y: 597.24999833107
description:
  text: Join tables on order_id, keep all records from the left table, exclude order_id from right table columns.
  hash: 8962dc93
previewCodeHash: 521251b3a0f8c770
previewMode: "1000"
config:
  join_type: left
  join_keys:
    - left: order_id
      right: order_id
  join_conditions: ""
  match_case: false
  left_columns:
    edits: []
    ordered: []
  right_columns:
    edits:
      - column: order_id
        checked: false
    ordered: []
input:
  - node: join_5
    input_port: left
    output_port: joined_data
  - node: python_4
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Any, Dict, List
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(side: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + side + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(df_left, df_right, left_cfg: Dict[str, Any], right_cfg: Dict[str, Any]):
    left_cols = list(df_left.columns)
    right_cols = list(df_right.columns)
    left_edits = _edits_by_column(left_cfg)
    right_edits = _edits_by_column(right_cfg)
    left_lower = {c.lower() for c in left_cols}

    out = []
    for name in _ordered_names(left_cols, left_cfg):
        edit = left_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("left", name)
        alias = edit.get("alias") if edit else None
        out.append(col.alias(alias) if alias else col.alias(name))
    for name in _ordered_names(right_cols, right_cfg):
        edit = right_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("right", name)
        alias = edit.get("alias") if edit else None
        if alias:
            out.append(col.alias(alias))
        elif name.lower() in left_lower:
            out.append(col.alias("right_" + name))
        else:
            out.append(col.alias(name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_keys: List[Dict[str, str]] = config.get("join_keys", [])
    join_condition = config.get("join_conditions", "")
    join_type = config.get("join_type") or "split_join"
    match_case = config.get("match_case", False)
    left_cfg = config.get("left_columns") or {}
    right_cfg = config.get("right_columns") or {}

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    left_types = {f.name.lower(): f.dataType for f in df_left.schema}
    right_types = {f.name.lower(): f.dataType for f in df_right.schema}

    def key_col(side, name, types):
        col = _qualified_col(side, name)
        if not match_case and isinstance(types.get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    predicates = []
    for key in join_keys:
        predicates.append(
            key_col("left", key["left"], left_types)
            == key_col("right", key["right"], right_types)
        )
    if join_condition:
        predicates.append(F.expr(join_condition))

    join_expr = None
    for predicate in predicates:
        join_expr = predicate if join_expr is None else join_expr & predicate

    is_split = join_type == "split_join"
    matched_how = "inner" if is_split else join_type

    if join_expr is None:
        matched = df_left.join(df_right, how=matched_how)
    else:
        matched = df_left.join(df_right, join_expr, how=matched_how)

    projection = _projection(df_left, df_right, left_cfg, right_cfg)
    if projection:
        matched = matched.select(*projection)

    if is_split:
        if join_expr is None:
            left_unmatched = df_left.join(df_right, how="left_anti")
            right_unmatched = df_right.join(df_left, how="left_anti")
        else:
            left_unmatched = df_left.join(df_right, join_expr, how="left_anti")
            right_unmatched = df_right.join(df_left, join_expr, how="left_anti")
    else:
        left_unmatched = spark.createDataFrame([], df_left.schema)
        right_unmatched = spark.createDataFrame([], df_right.schema)

    return {
        "joined_data": matched,
        "left_unmatched": left_unmatched,
        "right_unmatched": right_unmatched,
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_keys": [
        {
            "left": "order_id",
            "right": "order_id"
        }
    ],
    "join_conditions": "",
    "match_case": False,
    "left_columns": {
        "edits": [],
        "ordered": []
    },
    "right_columns": {
        "edits": [
            {
                "column": "order_id",
                "checked": False
            }
        ],
        "ordered": []
    }
}
inputs = {
    "left": ctx["join_5.joined_data"],
    "right": ctx["python_4.result"]
}
out = run(config, inputs, spark)
ctx["join_6.joined_data"] = out["joined_data"]
ctx["join_6.left_unmatched"] = out["left_unmatched"]
ctx["join_6.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False):
    display(ctx["join_6.joined_data"])
    display(ctx["join_6.left_unmatched"])
    display(ctx["join_6.right_unmatched"])

In [0]:
"""
id: aggregate_7
template: aggregate
templateVersion: 2.0.0
name: OrdersbyCity
position:
  x: 1966.9999561309814
  y: 510.24999737739563
description:
  text: Group data by city and calculate total orders and total amount.
  hash: d54ae58a
previewCodeHash: 6795fa4df0b7d9e7
previewMode: "1000"
config:
  group_bys:
    - expr: city
      type: column
  aggregations:
    - columnExpr:
        expr: order_id
        type: column
      fn: COUNT
      alias: TotalOrders
    - columnExpr:
        expr: ROUND(SUM(unit_price),2)
        type: expr
      fn: "-"
      alias: TotalAount
input:
  - node: join_6
    input_port: data
    output_port: joined_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "city",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "column"
            },
            "fn": "COUNT",
            "alias": "TotalOrders",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "ROUND(SUM(unit_price),2)",
                "type": "expr"
            },
            "fn": "-",
            "alias": "TotalAount",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["join_6.joined_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_7.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["aggregate_7.aggregated_data"])

In [0]:
"""
id: prepare_order_month
template: prepare
templateVersion: 1.0.0
name: Derive_order_month
position:
  x: 1860
  y: 795
description:
  text: Create a new column showing the year and month extracted from order date.
  hash: c28ff2bd
previewCodeHash: b81f33cabdb67949
previewMode: "1000"
config:
  actions:
    - type: formula
      target: order_month
      expression: substring(cast(order_date as string), 1, 7)
input:
  - node: join_6
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "order_month",
            "expression": "substring(cast(order_date as string), 1, 7)"
        }
    ]
}
inputs = {
    "data": ctx["join_6.joined_data"]
}
out = run(config, inputs, spark)
ctx["prepare_order_month.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["prepare_order_month.prepared_data"])

In [0]:
"""
id: sort_8
template: sort
templateVersion: 1.0.0
name: sorted
position:
  x: 2213.333281993866
  y: 508.24999737739563
description:
  text: Sort by TotalAount in descending order.
  hash: 8db6d0f2
previewCodeHash: 9bb2968ab3350967
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: TotalAount
        type: column
      sortBy: DESC
input:
  - node: aggregate_7
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "TotalAount",
                "type": "column"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_7.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_8.sorted_data"] = out["sorted_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["sort_8.sorted_data"])

In [0]:
"""
id: aggregate_order_status_month
template: aggregate
templateVersion: 2.0.0
name: Count_OrderStatus_by_Month
position:
  x: 2100
  y: 650
description:
  text: Group by order month and order status, then count the number of entries for each group.
  hash: 31b726a2
previewCodeHash: bb8b4cc5b6ce48e8
previewMode: "1000"
config:
  group_bys:
    - expr: order_month
      type: column
    - expr: order_status
      type: column
  aggregations:
    - columnExpr:
        expr: order_status
        type: column
      fn: COUNT
      alias: status_count
input:
  - node: prepare_order_month
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "order_month",
            "type": "column"
        },
        {
            "expr": "order_status",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_status",
                "type": "column"
            },
            "fn": "COUNT",
            "alias": "status_count",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["prepare_order_month.prepared_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_order_status_month.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["aggregate_order_status_month.aggregated_data"])

In [0]:
"""
id: output_9
template: output
templateVersion: 3.0.0
name: lakeflow_designer.enrich.AggregatedOrders
position:
  x: 2473.333281993866
  y: 508.24999737739563
description:
  text: Overwrite the AggregatedOrders table with sorted data in the specified catalog and schema.
  hash: 5c5ec3ea
previewMode: "1000"
config:
  output_type: table
  catalog: lakeflow_designer
  schema: enrich
  table_name: AggregatedOrders
  write_mode: overwrite
input:
  - node: sort_8
    input_port: data
    output_port: sorted_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "lakeflow_designer",
    "schema": "enrich",
    "table_name": "AggregatedOrders",
    "write_mode": "overwrite"
}
inputs = {
    "data": ctx["sort_8.sorted_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: sort_order_status_month
template: sort
templateVersion: 1.0.0
name: Sort_OrderStatus_by_Month_desc
position:
  x: 2350
  y: 650
description:
  text: Sort data by order month and order status in descending order.
  hash: 33a2360b
previewCodeHash: 5d12d3ecdfcb1826
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: order_month
        type: column
      sortBy: DESC
    - columnExpr:
        expr: order_status
        type: column
      sortBy: DESC
input:
  - node: aggregate_order_status_month
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "order_month",
                "type": "column"
            },
            "sortBy": "DESC"
        },
        {
            "columnExpr": {
                "expr": "order_status",
                "type": "column"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_order_status_month.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_order_status_month.sorted_data"] = out["sorted_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["sort_order_status_month.sorted_data"])

In [0]:
"""
id: output_13
template: output
templateVersion: 3.0.0
name: lakeflow_designer.enrich.AggregatedOrderStatus
position:
  x: 2610
  y: 650
description:
  text: Overwrite the table 'AggregatedOrderStatus' in the 'lakeflow_designer.enrich' schema with new data.
  hash: 339a7957
previewMode: "1000"
config:
  output_type: table
  catalog: lakeflow_designer
  schema: enrich
  table_name: AggregatedOrderStatus
  write_mode: overwrite
input:
  - node: sort_order_status_month
    input_port: data
    output_port: sorted_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "lakeflow_designer",
    "schema": "enrich",
    "table_name": "AggregatedOrderStatus",
    "write_mode": "overwrite"
}
inputs = {
    "data": ctx["sort_order_status_month.sorted_data"]
}
out = run(config, inputs, spark)